In [1]:
import pandas as pd
pd.set_option("display.width", 2000)
import matplotlib.pyplot as plt

In [3]:
# wczytanie recenzji 
pr_rev_df = pd.read_parquet("../pr_reviews.parquet") 
hu_pr_rev_df = pd.read_parquet("../human_pull_request_reviews.parquet")

In [4]:
# wyciągnięcie url repozytoriów zwierające ludzkie PRy
pr_hu_df = pd.read_parquet("../human_pull_request.parquet")
human_urls = pd.DataFrame({"url": pr_hu_df["repo_url"].unique()})

In [5]:
# PRy AI pochodzące z repo, w których są ludzkie PR-y
pr_df = pd.read_parquet("../pull_request.parquet")
union_human_bot = pr_df.merge(human_urls, left_on="repo_url", right_on="url")

In [6]:
# wzbogacenie informacji
bot_pr_rev = pr_rev_df.merge(union_human_bot, left_on="pr_id", right_on="id", suffixes=(None, "_x"))

In [7]:
# zmiana typu kolumny pr_id z object na int64
hu_pr_rev_df["pr_id"] = pd.to_numeric(hu_pr_rev_df["pr_id"], errors="coerce").astype("Int64")
# wzbogacenie informacji
hu_pr_rev = hu_pr_rev_df.merge(pr_hu_df, left_on="pr_id", right_on="id", suffixes=(None, "_x"))

In [8]:
# Reviews

In [9]:
# połączenie obu tabeli, aby ułatwić statystyki
bot_pr_rev["source"] = "AI"
hu_pr_rev["source"] = "human"

In [10]:
pd.set_option('display.max_colwidth', 70)
reviews = pd.concat([bot_pr_rev, hu_pr_rev], ignore_index=True)
reviews["body_len"] = reviews["body"].fillna("").apply(len)
reviews["user_type"] = reviews["user_type"].replace("Human", "User")
# reviews[reviews["source"] == "AI"][["id","user_type","body", "submitted_at", "html_url", "body_len"]].sort_values("body_len").head(50)
reviews[reviews["html_url"] == "https://github.com/Azure/azure-sdk-for-net/pull/50731"][["id","user_type","body", "submitted_at", "html_url", "body_len"]].sort_values("body_len").head(50)

,id,user_type,body,submitted_at,html_url,body_len
9064,2944035563,User,None,2025-06-19T21:15:31Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9066,2944038232,Bot,None,2025-06-19T21:17:50Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9067,2944078074,User,None,2025-06-19T21:53:06Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9068,2944087203,Bot,None,2025-06-19T21:58:20Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9070,2944475096,Bot,None,2025-06-20T03:48:48Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9069,2944470681,User,None,2025-06-20T03:44:41Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,0
9065,2944038059,User,need to generate with a newer commit id,2025-06-19T21:17:38Z,https://github.com/Azure/azure-sdk-for-net/pull/50731,39


In [11]:
# podstawowe statystyki
summary = reviews.groupby(["source", "user_type"]).agg(
    total_reviews=("id", "count"),
    unique_reviewers=("user", "nunique"),
    unique_prs=("pr_id", "nunique")
).reset_index()

summary

,source,user_type,total_reviews,unique_reviewers,unique_prs
0,AI,Bot,6434,27,2289
1,AI,User,10588,1333,3449
2,human,Bot,2434,30,1234
3,human,User,9259,1828,3323


In [12]:
# statystki statusu
state_counts = reviews.groupby(["source", "state",]).size().reset_index(name="count")

state_counts["percent"] = state_counts.groupby(["source"])["count"].transform(
    lambda x: 100 * x / x.sum()
)

state_counts

,source,state,count,percent
0,AI,APPROVED,4008,23.544616
1,AI,CHANGES_REQUESTED,793,4.658403
2,AI,COMMENTED,12009,70.545732
3,AI,DISMISSED,213,1.251248
4,human,APPROVED,3699,31.634311
5,human,CHANGES_REQUESTED,320,2.736680
6,human,COMMENTED,7462,63.815958
7,human,DISMISSED,212,1.813051


In [13]:
# statystki statusu w zależności od typu recenzenta
state_counts = reviews.groupby(["source", "user_type", "state",]).size().reset_index(name="count")

state_counts["percent"] = state_counts.groupby(["source", "user_type"])["count"].transform(
    lambda x: 100 * x / x.sum()
)

state_counts[state_counts["user_type"] != "Bot"]

,source,user_type,state,count,percent
4,AI,User,APPROVED,3998,37.759728
5,AI,User,CHANGES_REQUESTED,775,7.319607
6,AI,User,COMMENTED,5609,52.975066
7,AI,User,DISMISSED,206,1.945599
12,human,User,APPROVED,3630,39.205098
13,human,User,CHANGES_REQUESTED,291,3.142888
14,human,User,COMMENTED,5147,55.589156
15,human,User,DISMISSED,191,2.062858


In [14]:
# statystyki recenzji ogólne
reviews_per_pr = (
    reviews.groupby(["source", "user_type", "pr_id"])
    .size()
    .reset_index(name="reviews_count")
)

reviews_per_pr.groupby(["source", "user_type"])["reviews_count"].describe()

count      mean       std  min  25%  50%  75%   max
source user_type                                                      
AI     Bot        2289.0  2.810834  3.434266  1.0  1.0  1.0  3.0  30.0
       User       3449.0  3.069875  3.222157  1.0  1.0  2.0  4.0  30.0
human  Bot        1234.0  1.972447  2.482553  1.0  1.0  1.0  2.0  27.0
       User       3323.0  2.786338  3.824267  1.0  1.0  1.0  3.0  30.0

In [15]:
# statystyki długości recenzji
reviews["body_length"] = reviews["body"].fillna("").apply(len)
reviews.groupby(["source", "user_type"])["body_length"].describe()

count         mean          std  min     25%     50%      75%       max
source user_type                                                                           
AI     Bot         6434.0  1150.774324  3708.620399  0.0    0.00     0.0  1237.75  119903.0
       User       10588.0    27.413109   342.423804  0.0    0.00     0.0     0.00   29428.0
human  Bot         2434.0  4028.109696  9953.129552  0.0  206.25  1612.5  3868.50  158837.0
       User        9259.0    15.480613    93.563228  0.0    0.00     0.0     0.00    3043.0

In [16]:
# statystyki długości tytyłu
reviews["title_length"] = reviews["title"].fillna("").apply(len)
reviews.groupby(["source", "user_type"])["title_length"].describe()

count       mean        std   min   25%   50%   75%    max
source user_type                                                              
AI     Bot         6434.0  59.067299  19.866036   7.0  44.0  59.0  73.0  130.0
       User       10588.0  60.630431  18.418518  10.0  48.0  60.0  73.0  161.0
human  Bot         2434.0  42.603533  19.899939   3.0  28.0  40.0  55.0  169.0
       User        9259.0  46.848472  21.095060   4.0  31.0  45.0  60.0  389.0

In [17]:
# statystyki długości recenzji po uzwględnieniu tytyłu
reviews["body_title_length"] = reviews["body"].fillna("").apply(len) + reviews["title"].fillna("").apply(len)
reviews.groupby(["source", "user_type"])["body_title_length"].describe()

count         mean          std   min    25%     50%      75%       max
source user_type                                                                           
AI     Bot         6434.0  1209.841623  3704.659361   7.0   61.0    80.0  1292.25  119944.0
       User       10588.0    88.043540   342.800097  10.0   50.0    63.0    78.00   29473.0
human  Bot         2434.0  4070.713229  9953.151942   8.0  246.5  1659.5  3910.00  158872.0
       User        9259.0    62.329085    96.212896   4.0   33.0    48.0    66.00    3109.0

In [18]:
# jaki rozkład recenzentów (ogólny)
review_mix = (
    reviews.groupby(["source", "pr_id"])["user_type"]
    .unique()
    .reset_index()
)

def classify_mix(user_types):
    if len(user_types) == 1:
        if user_types[0] == "Bot":
            return "bot-only"
        else:
            return "human-only"
    else:
        return "mixed"

review_mix["review_mix_type"] = review_mix["user_type"].apply(classify_mix)

mix_summary = (
    review_mix.groupby(["source", "review_mix_type"])
    .size()
    .reset_index(name="count")
)

mix_summary["percent"] = mix_summary.groupby("source")["count"].transform(
    lambda x: 100 * x / x.sum()
)

mix_summary

,source,review_mix_type,count,percent
0,AI,bot-only,956,21.702611
1,AI,human-only,2115,48.013621
2,AI,mixed,1334,30.283768
3,human,bot-only,594,15.164667
4,human,human-only,2683,68.496298
5,human,mixed,640,16.339035


In [19]:
# jaki rozkład recenzentów (szczegółowy) UWAGA: wolno działa

review_mix_v2 = (
    reviews.groupby(["source", "pr_id"]).agg({
        "user_type": lambda x: list(x.unique()),
        'id': "count"
    })
    .reset_index()
    .rename(columns={"id": "review_count"})
)

def classify_mix_v2(row):
    user_types = row["user_type"]
    count = row["review_count"]
    
    if len(user_types) == 1:
        if user_types[0] == "Bot":
            return f"bot-only ({count})"
        else:
            return f"human-only ({count})"
    else:
        pr_rows = reviews[(reviews["source"] == row["source"]) & (reviews["pr_id"] == row["pr_id"])]
        bot_count = (pr_rows["user_type"] == "Bot").sum()
        human_count = (pr_rows["user_type"] != "Bot").sum()
        return f"mixed ({count}) (AI: {bot_count}, H: {human_count})"

    

review_mix_v2["review_mix_detailed"] = review_mix_v2.apply(classify_mix_v2, axis=1)

mix_detailed_summary = (
    review_mix_v2.groupby(["source", "review_mix_detailed"])
    .size()
    .reset_index(name="count")
)

mix_detailed_summary["percent"] = mix_detailed_summary.groupby("source")["count"].transform(
    lambda x: 100 * x / x.sum()
)

pd.set_option("display.max_rows", None)
mix_detailed_summary.sort_values(["source", "percent"], ascending=[True, False], inplace=True)

In [20]:
# najczęściej występujące słowa w recenzjach
from collections import Counter
import re
#import nltk
#from nltk.corpus import stopwords

stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

def analyze_common(df, source, user_type, top, isState, state = None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["body"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)

    return word_count.most_common(top)

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        if utype != None:
            top_words = analyze_common(reviews, src, utype, 20, False)
            
            print(f"Source: {src}, User type: {utype}")
            for word, count in top_words:
                print(f"{word}: {count}")
            print("==============================")
'''

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        for state in reviews["state"].unique():
            if utype != None:
                top_words = analyze_common(reviews, src, utype, 20, True, state)
                
                print(f"Source: {src}, User type: {utype}, Status: {state}")
                for word, count in top_words:
                    print(f"{word}: {count}")
                print("==============================")
  '''              

Source: AI, User type: User
src: 596
lgtm: 432
web: 301
components: 207
test: 198
good: 187
tests: 184
android: 184
error: 179
changes: 174
build: 170
should: 153
use: 153
please: 149
looks: 147
all: 146
tasks: 142
webeventcallbackfactoryeventargsextensions: 140
code: 137
https: 135
Source: AI, User type: Bot
summary: 23153
details: 22903
code: 8877
src: 7559
app: 6743
https: 5094
github: 4731
file: 4478
tsx: 4456
test: 4345
review: 4223
comment: 4013
blockquote: 3838
com: 3563
comments: 3133
dev: 3058
files: 3016
check: 2932
dashboard: 2821
cursor: 2685
Source: human, User type: User
lgtm: 432
thanks: 199
good: 179
looks: 157
https: 141
com: 136
github: 128
thank: 125
test: 98
just: 97
comments: 95
should: 94
tests: 87
some: 83
great: 81
would: 80
think: 78
nice: 76
work: 75
there: 73
Source: human, User type: Bot
details: 35694
summary: 35137
github: 12116
check: 11865
code: 11232
comment: 9957
src: 9594
tests: 8732
app: 8731
blockquote: 7126
test: 6413
trace: 6323
nox: 5909
hunks: 5

'\n\nfor src in reviews["source"].unique():\n    for utype in reviews["user_type"].unique():\n        for state in reviews["state"].unique():\n            if utype != None:\n                top_words = analyze_common(reviews, src, utype, 20, True, state)\n                \n                print(f"Source: {src}, User type: {utype}, Status: {state}")\n                for word, count in top_words:\n                    print(f"{word}: {count}")\n                print("==============================")\n  '

In [ ]:
from collections import Counter
import re
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

# funkcja do analizy najczestszych slow
def analyze_common(df, source, user_type, isState, state=None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["body"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)
    return word_count

# funkcja do tworzenia wykresu slupkowego
def plot_bar_word_cloud(word_count, title="", max_words=20, save_path=None):
    most_common = word_count.most_common(max_words)
    if len(most_common) == 0:
        return
    
    words = [w for w,f in most_common]
    freqs = [f for w,f in most_common]

    plt.figure(figsize=(12,6))
    plt.barh(words[::-1], freqs[::-1], color='skyblue')
    plt.title(title, fontsize=14)
    plt.xlabel("Count")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# katalogi do zapisu
os.makedirs("plots/review_body_without_state", exist_ok=True)
os.makedirs("csv/review_body_without_state", exist_ok=True)

max_words = 20  # Liczba najczestszych slow

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        if utype is None:
            continue
        word_count = analyze_common(reviews, src, utype, True, state)
        if len(word_count) == 0:
            continue

        title = f"Source: {src}, User: {utype}"
        safe_name = f"{src}_{utype}".replace(" ", "_").replace("/", "_")

        # Zapis wykresu PNG
        plot_path = f"plots/review_body_without_state/{safe_name}.png"
        plot_bar_word_cloud(word_count, title=title, max_words=max_words, save_path=plot_path)

        # Przygotowanie danych w formacie "weight","word","color"
        most_common = word_count.most_common(max_words)
        df_export = pd.DataFrame(most_common, columns=["word","weight"])
        df_export = df_export[["weight","word"]]  # kolejnosc kolumn
        df_export["color"] = ""  # pusta kolumna
        df_export["url"] = ""  # pusta kolumna
        df_export = df_export[["weight","word","color","url"]] 

        # Eksport do CSV z cudzyslowami
        csv_path = f"csv/review_body_without_state/{safe_name}.csv"
        df_export.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)
            
        print(f"Zapisano wykres: {plot_path} i CSV: {csv_path}")


In [28]:
from collections import Counter
import re
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

# funkcja do analizy najczęstszych slow
def analyze_common(df, source, user_type, isState, state=None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["body"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)
    return word_count

# funkcja do tworzenia wykresu słupkowego
def plot_bar_word_cloud(word_count, title="", max_words=20, save_path=None):
    most_common = word_count.most_common(max_words)
    if len(most_common) == 0:
        return
    
    words = [w for w,f in most_common]
    freqs = [f for w,f in most_common]

    plt.figure(figsize=(12,6))
    plt.barh(words[::-1], freqs[::-1], color='skyblue')
    plt.title(title, fontsize=14)
    plt.xlabel("Count")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# katalogi do zapisu
os.makedirs("plots/review_body_with_state", exist_ok=True)
os.makedirs("csv/review_body_with_state", exist_ok=True)

max_words = 20  # Liczba najczęstszych słów

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        for state in reviews["state"].unique():
            if utype is None:
                continue
            word_count = analyze_common(reviews, src, utype, True, state)
            if len(word_count) == 0:
                continue

            title = f"Source: {src}, User: {utype}, Status: {state}"
            safe_name = f"{src}_{utype}_{state}".replace(" ", "_").replace("/", "_")

            # Zapis wykresu PNG
            plot_path = f"plots/review_body_with_state/{safe_name}.png"
            plot_bar_word_cloud(word_count, title=title, max_words=max_words, save_path=plot_path)

            # Przygotowanie danych w formacie "weight","word","color"
            most_common = word_count.most_common(max_words)
            df_export = pd.DataFrame(most_common, columns=["word","weight"])
            df_export = df_export[["weight","word"]]  # kolejność kolumn
            df_export["color"] = ""  # pusta kolumna
            df_export["url"] = ""  # pusta kolumna
            df_export = df_export[["weight","word","color","url"]]  # ostateczny układ

            # Eksport do CSV z cudzysłowami
            csv_path = f"csv/review_body_with_state/{safe_name}.csv"
            df_export.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)
            
            print(f"Zapisano wykres: {plot_path} i CSV: {csv_path}")


Zapisano wykres: plots/review_body_with_state/AI_User_COMMENTED.png i CSV: csv/review_body_with_state/AI_User_COMMENTED.csv
Zapisano wykres: plots/review_body_with_state/AI_User_APPROVED.png i CSV: csv/review_body_with_state/AI_User_APPROVED.csv
Zapisano wykres: plots/review_body_with_state/AI_User_CHANGES_REQUESTED.png i CSV: csv/review_body_with_state/AI_User_CHANGES_REQUESTED.csv
Zapisano wykres: plots/review_body_with_state/AI_User_DISMISSED.png i CSV: csv/review_body_with_state/AI_User_DISMISSED.csv
Zapisano wykres: plots/review_body_with_state/AI_Bot_COMMENTED.png i CSV: csv/review_body_with_state/AI_Bot_COMMENTED.csv
Zapisano wykres: plots/review_body_with_state/AI_Bot_APPROVED.png i CSV: csv/review_body_with_state/AI_Bot_APPROVED.csv
Zapisano wykres: plots/review_body_with_state/AI_Bot_CHANGES_REQUESTED.png i CSV: csv/review_body_with_state/AI_Bot_CHANGES_REQUESTED.csv
Zapisano wykres: plots/review_body_with_state/AI_Bot_DISMISSED.png i CSV: csv/review_body_with_state/AI_Bot_DI

In [19]:
# najczęściej występujące słowa w tytułach recenzji
from collections import Counter
import re
#import nltk
#from nltk.corpus import stopwords

stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

def analyze_common(df, source, user_type, top, isState, state = None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["title"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)

    return word_count.most_common(top)

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        if utype != None:
            top_words = analyze_common(reviews, src, utype, 20, False)
            
            print(f"Source: {src}, User type: {utype}")
            for word, count in top_words:
                print(f"{word}: {count}")
            print("==============================")
'''

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        for state in reviews["state"].unique():
            if utype != None:
                top_words = analyze_common(reviews, src, utype, 20, True, state)
                
                print(f"Source: {src}, User type: {utype}, Status: {state}")
                for word, count in top_words:
                    print(f"{word}: {count}")
                print("==============================")
          '''      

Source: AI, User type: User
add: 3254
fix: 2561
feat: 1381
update: 775
remove: 689
support: 577
implement: 568
documentation: 512
test: 463
docs: 455
refactor: 413
azure: 366
chore: 364
tests: 346
improve: 334
use: 324
net: 261
error: 255
all: 252
version: 250
Source: AI, User type: Bot
add: 2063
fix: 1386
feat: 488
update: 484
support: 447
implement: 415
remove: 308
test: 305
documentation: 286
tests: 282
azure: 273
improve: 241
docs: 200
net: 197
chore: 190
page: 178
use: 166
comprehensive: 163
files: 159
cli: 159
Source: AI, User type: Human
Source: human, User type: User
Source: human, User type: Bot
add: 531
feat: 497
fix: 428
chore: 184
update: 153
api: 145
docs: 107
support: 105
bump: 94
agents: 93
code: 76
config: 59
project: 59
new: 57
refactor: 56
test: 56
added: 56
tests: 55
remove: 55
release: 55
Source: human, User type: Human
fix: 1727
feat: 1602
add: 1580
support: 643
update: 635
chore: 469
docs: 359
test: 298
api: 277
use: 240
remove: 219
implement: 217
refactor: 216
ve

'\n\nfor src in reviews["source"].unique():\n    for utype in reviews["user_type"].unique():\n        for state in reviews["state"].unique():\n            if utype != None:\n                top_words = analyze_common(reviews, src, utype, 20, True, state)\n                \n                print(f"Source: {src}, User type: {utype}, Status: {state}")\n                for word, count in top_words:\n                    print(f"{word}: {count}")\n                print("==============================")\n          '

In [29]:
from collections import Counter
import re
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

# funkcja do analizy najczestszych slow
def analyze_common(df, source, user_type, isState, state=None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["title"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)
    return word_count

# funkcja do tworzenia wykresu slupkowego
def plot_bar_word_cloud(word_count, title="", max_words=20, save_path=None):
    most_common = word_count.most_common(max_words)
    if len(most_common) == 0:
        return
    
    words = [w for w,f in most_common]
    freqs = [f for w,f in most_common]

    plt.figure(figsize=(12,6))
    plt.barh(words[::-1], freqs[::-1], color='skyblue')
    plt.title(title, fontsize=14)
    plt.xlabel("Count")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# katalogi do zapisu
os.makedirs("plots/review_title_with_state", exist_ok=True)
os.makedirs("csv/review_title_with_state", exist_ok=True)

max_words = 20  # Liczba najczestszych slow

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        for state in reviews["state"].unique():
            if utype is None:
                continue
            word_count = analyze_common(reviews, src, utype, True, state)
            if len(word_count) == 0:
                continue

            title = f"Source: {src}, User: {utype}, Status: {state}"
            safe_name = f"{src}_{utype}_{state}".replace(" ", "_").replace("/", "_")

            # Zapis wykresu PNG
            plot_path = f"plots/review_title_with_state/{safe_name}.png"
            plot_bar_word_cloud(word_count, title=title, max_words=max_words, save_path=plot_path)

            # Przygotowanie danych w formacie "weight","word","color"
            most_common = word_count.most_common(max_words)
            df_export = pd.DataFrame(most_common, columns=["word","weight"])
            df_export = df_export[["weight","word"]]  # kolejnosc kolumn
            df_export["color"] = ""  # pusta kolumna
            df_export["url"] = ""  # pusta kolumna
            df_export = df_export[["weight","word","color","url"]] 

            # Eksport do CSV z cudzyslowami
            csv_path = f"csv/review_title_with_state/{safe_name}.csv"
            df_export.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)
            
            print(f"Zapisano wykres: {plot_path} i CSV: {csv_path}")


Zapisano wykres: plots/review_title_with_state/AI_User_COMMENTED.png i CSV: csv/review_title_with_state/AI_User_COMMENTED.csv
Zapisano wykres: plots/review_title_with_state/AI_User_APPROVED.png i CSV: csv/review_title_with_state/AI_User_APPROVED.csv
Zapisano wykres: plots/review_title_with_state/AI_User_CHANGES_REQUESTED.png i CSV: csv/review_title_with_state/AI_User_CHANGES_REQUESTED.csv
Zapisano wykres: plots/review_title_with_state/AI_User_DISMISSED.png i CSV: csv/review_title_with_state/AI_User_DISMISSED.csv
Zapisano wykres: plots/review_title_with_state/AI_Bot_COMMENTED.png i CSV: csv/review_title_with_state/AI_Bot_COMMENTED.csv
Zapisano wykres: plots/review_title_with_state/AI_Bot_APPROVED.png i CSV: csv/review_title_with_state/AI_Bot_APPROVED.csv
Zapisano wykres: plots/review_title_with_state/AI_Bot_CHANGES_REQUESTED.png i CSV: csv/review_title_with_state/AI_Bot_CHANGES_REQUESTED.csv
Zapisano wykres: plots/review_title_with_state/AI_Bot_DISMISSED.png i CSV: csv/review_title_with

In [30]:
from collections import Counter
import re
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv

# stop words
stop_words = {
    "the", "a", "an", "and", "or", "is", "it", "to", "in", "of", "for", "on", "this",
    "that", "with", "as", "by", "from", "are", "was", "be", "at", "which", "have",
    "has", "had", "will", "can", "not", "if", "else", "when", "then", "about", "up",
    "out", "so", "no", "yes", "your", "our", "their", "its", "but", "we", "you", "they"
}

# funkcja do analizy najczestszych slow
def analyze_common(df, source, user_type, isState, state=None):
    if isState:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type) & (df["state"] == state)] 
    else:
        subset = df[(df["source"] == source) & (df["user_type"] == user_type)] 
        
    all_text = " ".join(subset["title"].dropna().astype(str)).lower()
    words = re.findall(r"\b\w+\b", all_text)
    words = [w for w in words if w not in stop_words and len(w) > 2]
    word_count = Counter(words)
    return word_count

# funkcja do tworzenia wykresu slupkowego
def plot_bar_word_cloud(word_count, title="", max_words=20, save_path=None):
    most_common = word_count.most_common(max_words)
    if len(most_common) == 0:
        return
    
    words = [w for w,f in most_common]
    freqs = [f for w,f in most_common]

    plt.figure(figsize=(12,6))
    plt.barh(words[::-1], freqs[::-1], color='skyblue')
    plt.title(title, fontsize=14)
    plt.xlabel("Count")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# katalogi do zapisu
os.makedirs("plots/review_title_without_state", exist_ok=True)
os.makedirs("csv/review_title_without_state", exist_ok=True)

max_words = 20  # Liczba najczestszych slow

for src in reviews["source"].unique():
    for utype in reviews["user_type"].unique():
        if utype is None:
            continue
        word_count = analyze_common(reviews, src, utype, True, state)
        if len(word_count) == 0:
            continue

        title = f"Source: {src}, User: {utype}"
        safe_name = f"{src}_{utype}".replace(" ", "_").replace("/", "_")

        # Zapis wykresu PNG
        plot_path = f"plots/review_title_without_state/{safe_name}.png"
        plot_bar_word_cloud(word_count, title=title, max_words=max_words, save_path=plot_path)

        # Przygotowanie danych w formacie "weight","word","color"
        most_common = word_count.most_common(max_words)
        df_export = pd.DataFrame(most_common, columns=["word","weight"])
        df_export = df_export[["weight","word"]]  # kolejnosc kolumn
        df_export["color"] = ""  # pusta kolumna
        df_export["url"] = ""  # pusta kolumna
        df_export = df_export[["weight","word","color","url"]] 

        # Eksport do CSV z cudzyslowami
        csv_path = f"csv/review_title_without_state/{safe_name}.csv"
        df_export.to_csv(csv_path, index=False, quoting=csv.QUOTE_ALL)
            
        print(f"Zapisano wykres: {plot_path} i CSV: {csv_path}")


Zapisano wykres: plots/review_title_without_state/AI_User.png i CSV: csv/review_title_without_state/AI_User.csv
Zapisano wykres: plots/review_title_without_state/AI_Bot.png i CSV: csv/review_title_without_state/AI_Bot.csv
Zapisano wykres: plots/review_title_without_state/human_User.png i CSV: csv/review_title_without_state/human_User.csv
Zapisano wykres: plots/review_title_without_state/human_Bot.png i CSV: csv/review_title_without_state/human_Bot.csv
